In [0]:
%sql
USE CATALOG test;
USE SCHEMA sql;

DROP TABLE IF EXISTS population;
CREATE TABLE population (
    country VARCHAR(50),
    continent VARCHAR(10),
    total_population BIGINT
);

INSERT INTO population (country, continent, total_population)
VALUES
('India', 'Asia', 1428627663),
('China', 'Asia', 1425671352),
('United States', 'Americas', 339996564),
('Indonesia', 'Asia', 277534123),
('Pakistan', 'Asia', 240485658),
('Nigeria', 'Africa', 223804632),
('Brazil', 'Americas', 216422446),
('Bangladesh', 'Asia', 172954319),
('Russia', 'Europe', 144444359),
('Mexico', 'Americas', 128455567);

num_affected_rows,num_inserted_rows
10,10


In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("country", StringType(), True),
    StructField("continent", StringType(), True),
    StructField("total_population", LongType(), True)
])
data = [
    ("India", "Asia", 1428627663),
    ("China", "Asia", 1425671352),
    ("United States", "Americas", 339996564),
    ("Indonesia", "Asia", 277534123),
    ("Pakistan", "Asia", 240485658),
    ("Nigeria", "Africa", 223804632),
    ("Brazil", "Americas", 216422446),
    ("Bangladesh", "Asia", 172954319),
    ("Russia", "Europe", 144444359),
    ("Mexico", "Americas", 128455567)
]
population_df = spark.createDataFrame(data, schema)
population_df.write.mode("overwrite").saveAsTable("test.pyspark.population")

population_df = spark.read.table("test.pyspark.population")

In [0]:
%sql
select * from population;

country,continent,total_population
India,Asia,1428627663
China,Asia,1425671352
United States,Americas,339996564
Indonesia,Asia,277534123
Pakistan,Asia,240485658
Nigeria,Africa,223804632
Brazil,Americas,216422446
Bangladesh,Asia,172954319
Russia,Europe,144444359
Mexico,Americas,128455567


In [0]:
%sql
-- Find the countries with a population higher than the average population of the top 10 populated countries
select * from population where total_population > (select avg(total_population) from population);
select * from population where total_population = (select max(total_population) from population);

-- Find the population of countries, which are the only countries for their continent amongst the top 10 most populated countries.
select * from population where continent in (select continent from population group by continent having count(country) = 1);

-- Get population information of countries with highest population in each continent
select * from population where (continent, total_population) in (select continent, max(total_population) from population group by continent);

country,continent,total_population
India,Asia,1428627663
United States,Americas,339996564
Nigeria,Africa,223804632
Russia,Europe,144444359


In [0]:
from pyspark.sql.functions import avg, max, col, count

avg_pop = population_df.select(
    avg("total_population")
).collect()[0][0]

max_pop = population_df.select(
    max("total_population")
).collect()[0][0]

population_df.filter(
    col("total_population") > avg_pop
).show()

population_df.filter(
    col("total_population") == max_pop
).show()


# Find the population of countries, which are the only countries for their continent amongst the top 10 most populated countries.
a = population_df.groupBy(
    col("continent").alias("con")
).agg(
    count("country").alias("count_country"),
).filter(
    col("count_country") == 1
)

b = population_df.alias("b")
b.join(
    a,
    a.con == b.continent
).select(
    b.country,
    b.continent,
    b.total_population 
).display()




# Get population information of countries with highest population in each continent
a = population_df.groupBy(
    col("continent").alias("con")
).agg(
    max("total_population").alias("max")
)
b = population_df.alias("b")
b.join(
    a,
    (a.con == b.continent) & (a.max == b.total_population)
).select(
    b.country,
    b.continent,
    b.total_population 
).display()



+-------+---------+----------------+
|country|continent|total_population|
+-------+---------+----------------+
|  India|     Asia|      1428627663|
|  China|     Asia|      1425671352|
+-------+---------+----------------+

+-------+---------+----------------+
|country|continent|total_population|
+-------+---------+----------------+
|  India|     Asia|      1428627663|
+-------+---------+----------------+



country,continent,total_population
Nigeria,Africa,223804632
Russia,Europe,144444359


country,continent,total_population
India,Asia,1428627663
United States,Americas,339996564
Nigeria,Africa,223804632
Russia,Europe,144444359
